# 论文 16：用于关系推理的简单神经网络模块
## Adam Santoro, David Raposo, David G. T. Barrett et al., DeepMind（2017）

### 关系网络 (RN)

用于推理对象之间关系的即插即用模块。关键见解：显式计算成对关系！

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from itertools import combinations

np.random.seed(42)

## 关系网络架构

核心理念：
```
RN(O) = f_φ( Σ_{i,j} g_θ(o_i, o_j, q) )
```

- **g_θ**：关系函数，负责处理对象对
- **f_φ**：聚合函数（处理关系）
- **O**：对象集
- **q**：查询/上下文

In [ ]:
def relu(x):
    return np.maximum(0, x)

class MLP:
    '简单的多层感知器'
    def __init__(self, input_dim, hidden_dims, output_dim):
        self.layers = []
        
        # 创建图层
        dims = [input_dim] + hidden_dims + [output_dim]
        for i in range(len(dims) - 1):
            W = np.random.randn(dims[i+1], dims[i]) * 0.01
            b = np.zeros((dims[i+1], 1))
            self.layers.append((W, b))
    
    def forward(self, x):
        '通过 MLP 前向传递'
        if len(x.shape) == 1:
            x = x.reshape(-1, 1)
        
        for i, (W, b) in enumerate(self.layers):
            x = np.dot(W, x) + b
            # ReLU 用于除最后一层之外的所有层
            if i < len(self.layers) - 1:
                x = relu(x)
        
        return x.flatten()

# 测试多层感知机
mlp = MLP(input_dim=10, hidden_dims=[20, 20], output_dim=5)
test_input = np.random.randn(10)
output = mlp.forward(test_input)
print(f"MLP output shape: {output.shape}")

## 关系网络模块

In [ ]:
class RelationNetwork:
    """用于推理对象关系的关系网络
    
    RN(O) = f_φ( Σ_{i,j} g_θ(o_i, o_j, q) )"""
    def __init__(self, object_dim, query_dim, g_hidden_dims, f_hidden_dims, output_dim):
        """object_dim：每个对象表示的维度
        query_dim：查询/问题的维度
        g_hidden_dims：g_θ 的隐藏维度（关系函数）
        f_hidden_dims：f_φ 的隐藏维度（聚合函数）
        output_dim：最终输出尺寸"""
        # g_θ：处理对象对+查询
        g_input_dim = object_dim * 2 + query_dim
        g_output_dim = g_hidden_dims[-1] if g_hidden_dims else 256
        self.g_theta = MLP(g_input_dim, g_hidden_dims[:-1], g_output_dim)
        
        # f_φ：处理聚合关系
        f_input_dim = g_output_dim
        self.f_phi = MLP(f_input_dim, f_hidden_dims, output_dim)
    
    def forward(self, objects, query):
        """objects：对象表示列表（每个都是一个向量）
        query：查询/上下文向量
        
        返回：输出向量"""
        n_objects = len(objects)
        
        # 计算所有对的关系
        relations = []
        
        for i in range(n_objects):
            for j in range(n_objects):
                # 连接对象对+查询
                pair_input = np.concatenate([objects[i], objects[j], query])
                
                # 应用 g_θ 来计算关系
                relation = self.g_theta.forward(pair_input)
                relations.append(relation)
        
        # 聚合关系（总和）
        aggregated = np.sum(relations, axis=0)
        
        # 应用 f_φ 以获得最终输出
        output = self.f_phi.forward(aggregated)
        
        return output

# 创建关系网络
rn = RelationNetwork(
    object_dim=8,
    query_dim=4,
    g_hidden_dims=[32, 32, 32],
    f_hidden_dims=[64, 32],
    output_dim=10  # 例如 10 个答案类别
)

# 使用样本对象进行测试
test_objects = [np.random.randn(8) for _ in range(5)]
test_query = np.random.randn(4)

output = rn.forward(test_objects, test_query)
print(f"\nRelation Network output: {output[:5]}...")
print(f"Output shape: {output.shape}")

## CLEVR 数据集

使用彩色形状简化视觉推理任务

In [ ]:
class SortOfCLEVR:
    '生成 Sort-of-CLEVR 数据集'
    def __init__(self):
        self.colors = ['red', 'blue', 'green', 'orange', 'yellow', 'purple']
        self.shapes = ['circle', 'square', 'triangle']
        self.sizes = ['small', 'large']
    
    def generate_scene(self, n_objects=6):
        """生成带有对象的场景
        每个 object：（x、y、color_idx、shape_idx、size_idx）"""
        objects = []
        used_colors = set()
        
        for i in range(n_objects):
            # 随机位置
            x = np.random.uniform(0, 1)
            y = np.random.uniform(0, 1)
            
            # 独特的色彩
            available_colors = [c for c in range(len(self.colors)) if c not in used_colors]
            if not available_colors:
                break
            color_idx = np.random.choice(available_colors)
            used_colors.add(color_idx)
            
            # 形状和尺寸随机
            shape_idx = np.random.randint(len(self.shapes))
            size_idx = np.random.randint(len(self.sizes))
            
            objects.append({
                'x': x,
                'y': y,
                'color': color_idx,
                'shape': shape_idx,
                'size': size_idx
            })
        
        return objects
    
    def generate_question(self, scene, question_type='relational'):
        """生成questions：
        - 非 relational：“红色物体的形状是什么？”
        - Relational：“最接近红色物体的形状是什么？”"""
        if question_type == 'relational':
            # 选择一个参考对象
            ref_obj = np.random.choice(scene)
            
            # 找到最近的物体
            min_dist = float('inf')
            closest_obj = None
            for obj in scene:
                if obj is ref_obj:
                    continue
                dist = np.sqrt((obj['x'] - ref_obj['x'])**2 + (obj['y'] - ref_obj['y'])**2)
                if dist < min_dist:
                    min_dist = dist
                    closest_obj = obj
            
            question = f"Shape of object closest to {self.colors[ref_obj['color']]}?"
            answer = closest_obj['shape']
            
        else:  # 非关系型
            # 随机选择一个物体
            obj = np.random.choice(scene)
            question = f"What is the shape of the {self.colors[obj['color']]} object?"
            answer = obj['shape']
        
        return question, answer, question_type

# 生成示例场景
dataset = SortOfCLEVR()
scene = dataset.generate_scene(n_objects=6)

print("Generated scene:")
for i, obj in enumerate(scene):
    print(f"  Object {i}: {dataset.colors[obj['color']]:8s} "
          f"{dataset.shapes[obj['shape']]:8s} {dataset.sizes[obj['size']]:6s} "
          f"at ({obj['x']:.2f}, {obj['y']:.2f})")

# 生成问题
print("\nSample questions:")
for qtype in ['non-relational', 'relational', 'relational']:
    q, a, t = dataset.generate_question(scene, qtype)
    print(f"  [{t:15s}] {q}")
    print(f"  Answer: {dataset.shapes[a]}")

## 可视化场景

In [ ]:
def visualize_scene(scene, dataset):
    '可视化 CLEVR 场景'
    fig, ax = plt.subplots(figsize=(10, 10))
    
    # 颜色映射
    color_map = {
        'red': 'red',
        'blue': 'blue',
        'green': 'green',
        'orange': 'orange',
        'yellow': 'yellow',
        'purple': 'purple'
    }
    
    for obj in scene:
        x, y = obj['x'], obj['y']
        color = color_map[dataset.colors[obj['color']]]
        shape = dataset.shapes[obj['shape']]
        size = 300 if obj['size'] == 1 else 150
        
        if shape == 'circle':
            ax.scatter([x], [y], s=size, c=color, marker='o', edgecolors='black', linewidths=2)
        elif shape == 'square':
            ax.scatter([x], [y], s=size, c=color, marker='s', edgecolors='black', linewidths=2)
        else:  # 三角形
            ax.scatter([x], [y], s=size, c=color, marker='^', edgecolors='black', linewidths=2)
    
    ax.set_xlim(-0.1, 1.1)
    ax.set_ylim(-0.1, 1.1)
    ax.set_aspect('equal')
    ax.set_title('Sort-of-CLEVR Scene', fontsize=14, fontweight='bold')
    ax.grid(True, alpha=0.3)
    plt.show()

visualize_scene(scene, dataset)

## 对象表示编码器

In [ ]:
def encode_object(obj, dataset):
    """将对象编码为 vector：
    输出格式：[x, y, color_one_hot, shape_one_hot, size_one_hot]"""
    # 位置
    pos = np.array([obj['x'], obj['y']])
    
    # 1.1.1 独热编码
    color_oh = np.zeros(len(dataset.colors))
    color_oh[obj['color']] = 1
    
    shape_oh = np.zeros(len(dataset.shapes))
    shape_oh[obj['shape']] = 1
    
    size_oh = np.zeros(len(dataset.sizes))
    size_oh[obj['size']] = 1
    
    # 连接
    encoding = np.concatenate([pos, color_oh, shape_oh, size_oh])
    return encoding

def encode_question(question_text, ref_color, dataset):
    """将问题编码为向量（简化）
    在 practice 中：使用 LSTM 或嵌入"""
    # One-hot 参考色
    color_oh = np.zeros(len(dataset.colors))
    if ref_color is not None:
        color_oh[ref_color] = 1
    
    # 问题类型（简体：1 为关系型，0 为非关系型）
    is_relational = 1.0 if 'closest' in question_text else 0.0
    
    return np.concatenate([color_oh, [is_relational]])

# 测试编码
obj_encoding = encode_object(scene[0], dataset)
print(f"Object encoding shape: {obj_encoding.shape}")
print(f"Object encoding: {obj_encoding}")

q_encoding = encode_question("Shape of object closest to red?", 0, dataset)
print(f"\nQuestion encoding shape: {q_encoding.shape}")

## 完整流程：场景→对象→RN→答案

In [ ]:
# 创建具有正确尺寸的关系网络
object_dim = 2 + len(dataset.colors) + len(dataset.shapes) + len(dataset.sizes)
query_dim = len(dataset.colors) + 1

rn_visual = RelationNetwork(
    object_dim=object_dim,
    query_dim=query_dim,
    g_hidden_dims=[64, 64, 32],
    f_hidden_dims=[64, 32],
    output_dim=len(dataset.shapes)  # 预测形状
)

# 编码场景
encoded_objects = [encode_object(obj, dataset) for obj in scene]

# 生成问题
question, answer, qtype = dataset.generate_question(scene, 'relational')

# 从问题中提取参考颜色（简化）
ref_color = None
for i, color in enumerate(dataset.colors):
    if color in question.lower():
        ref_color = i
        break

encoded_question = encode_question(question, ref_color, dataset)

# 运行关系网络
prediction = rn_visual.forward(encoded_objects, encoded_question)
predicted_shape = np.argmax(prediction)

print(f"Question: {question}")
print(f"True answer: {dataset.shapes[answer]}")
print(f"Predicted answer: {dataset.shapes[predicted_shape]}")
print(f"\n(Model is untrained, so random prediction)")

## 可视化对象之间的关系

In [ ]:
# 计算成对距离（关系示例）
n_objects = len(scene)
distance_matrix = np.zeros((n_objects, n_objects))

for i in range(n_objects):
    for j in range(n_objects):
        dist = np.sqrt((scene[i]['x'] - scene[j]['x'])**2 + 
                      (scene[i]['y'] - scene[j]['y'])**2)
        distance_matrix[i, j] = dist

# 可视化
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# 有联系的场景
color_map = {'red': 'red', 'blue': 'blue', 'green': 'green', 
            'orange': 'orange', 'yellow': 'yellow', 'purple': 'purple'}

for i, obj_i in enumerate(scene):
    for j, obj_j in enumerate(scene):
        if i != j:
            # 绘制连接（较粗=较紧密）
            dist = distance_matrix[i, j]
            alpha = np.exp(-dist * 2)  # 较近的物体 = 较高的 alpha
            ax1.plot([obj_i['x'], obj_j['x']], [obj_i['y'], obj_j['y']], 
                    'k-', alpha=alpha, linewidth=1)

for obj in scene:
    color = color_map[dataset.colors[obj['color']]]
    ax1.scatter([obj['x']], [obj['y']], s=300, c=color, 
               edgecolors='black', linewidths=3, zorder=5)
    ax1.text(obj['x'], obj['y']-0.08, dataset.colors[obj['color']], 
            ha='center', fontsize=9, fontweight='bold')

ax1.set_xlim(-0.1, 1.1)
ax1.set_ylim(-0.2, 1.1)
ax1.set_aspect('equal')
ax1.set_title('Object Relations (spatial)', fontsize=14, fontweight='bold')
ax1.grid(True, alpha=0.3)

# 距离矩阵
im = ax2.imshow(distance_matrix, cmap='viridis')
ax2.set_xlabel('Object', fontsize=12)
ax2.set_ylabel('Object', fontsize=12)
ax2.set_title('Pairwise Distances', fontsize=14, fontweight='bold')
plt.colorbar(im, ax=ax2, label='Distance')

plt.tight_layout()
plt.show()

print(f"\nRelation Network considers ALL {n_objects * (n_objects - 1)} pairs!")

## 置换不变性检验

In [ ]:
# 测试 RN 对对象顺序不变
test_objects = [np.random.randn(object_dim) for _ in range(4)]
test_query = np.random.randn(query_dim)

# 原始订单
output1 = rn_visual.forward(test_objects, test_query)

# 打乱顺序
shuffled_objects = test_objects.copy()
np.random.shuffle(shuffled_objects)
output2 = rn_visual.forward(shuffled_objects, test_query)

# 检查输出是否相同
diff = np.linalg.norm(output1 - output2)

print("Permutation Invariance Test:")
print(f"Original output: {output1[:4]}...")
print(f"Shuffled output: {output2[:4]}...")
print(f"Difference: {diff:.10f}")
print(f"\n{'✓ PASSED' if diff < 1e-10 else '✗ FAILED'}: RN is permutation invariant!")

## 与基线比较（无关系推理）

In [ ]:
class BaselineNetwork:
    'Baseline：只是连接所有对象+查询，没有显式关系'
    def __init__(self, object_dim, query_dim, max_objects, output_dim):
        # 连接所有对象+查询
        input_dim = object_dim * max_objects + query_dim
        self.mlp = MLP(input_dim, [128, 64], output_dim)
        self.max_objects = max_objects
        self.object_dim = object_dim
    
    def forward(self, objects, query):
        # 填充或截断为 max_objects
        padded = []
        for i in range(self.max_objects):
            if i < len(objects):
                padded.append(objects[i])
            else:
                padded.append(np.zeros(self.object_dim))
        
        # 连接一切
        concat = np.concatenate(padded + [query])
        return self.mlp.forward(concat)

# 创建基线
baseline = BaselineNetwork(object_dim, query_dim, max_objects=10, output_dim=len(dataset.shapes))

# 测试
baseline_output = baseline.forward(encoded_objects, encoded_question)

print("Baseline Network (no explicit relations):")
print(f"Output: {baseline_output}")
print(f"\nBaseline doesn't explicitly reason about pairs!")

## 要点

### 关系网络（RN）公式：

$$
\text{RN}(O) = f_\phi \left( \sum_{i,j} g_\theta(o_i, o_j, q) \right)
$$

其中：
- $O = \{o_1, o_2, ..., o_n\}$：对象集
- $g_\theta$：关系函数（MLP），对对象对进行推理
- $f_\phi$：聚合函数（MLP），组合所有关系表示
- $q$：查询或上下文，例如问题

### 主要特性：

1. **显式的成对关系**：
   - 考虑所有 $n^2$ 对（或 $\binom{n}{2}$ 唯一对）
   - 每对均由 $g_\theta$ 独立处理

2. **置换不变性**：
   - 求和→顺序并不重要
   - $\text{RN}(\{o_1, o_2\}) = \text{RN}(\{o_2, o_1\})$

3. **可组合性**：
   - 可以作为模块接入不同架构
   - 对象表示可以来自 CNN、LSTM 等模型

### 架构细节：

**对于视觉问答**：
```
Image → CNN → Feature maps → Objects (spatial positions)
Question → LSTM → Query embedding
Objects + Query → RN → Answer
```

**对于文本**：
```
Sentence → LSTM → Word embeddings → Objects
Query → Embedding
Objects + Query → RN → Answer
```

### 计算复杂度：

- **对**：$O(n^2)$，其中 $n$ = 对象数量
- **g_θ 计算**：需要执行 $n^2$ 次前向传播
- 对于大型 $n$ 可能会很昂贵
- 可以使用 $i \neq j$ 排除自对 → $n(n-1)$ 对

### 结果：

**Sort-of-CLEVR**：
- 关系问题：96% (RN) vs 63%（CNN 基线）
- 非关系型：98% (RN) vs 98% (CNN)

**CLEVR**（完整数据集）：
- 95.5% 的准确率（超人的表现！）
- 历届最佳成绩：68.5%

**bAbI**：
- 单一模型的 18/20 任务
- 在关系推理任务上表现出色

### 为什么它有效：

1. **归纳偏置**：显式建模关系
2. **数据效率**：结构化计算→所需数据更少
3. **可解释性**：可以可视化 $g_\theta$ 输出
4. **泛化**：学习关系模式

### 与其他方法的比较：

| 方法 | 成对关系 | 置换不变性 | 复杂度 |
|----------|-------------------|----------------------|------------|
| CNN |隐式| ✗ | $O(n)$ |
| RNN/LSTM |顺序 | ✗ | $O(n)$ |
| 注意力 | 加权对象对 | ✓ | $O(n^2)$ |
| **RN** | **显式** | **✓** | **$O(n^2)$** |
| 图神经网络 | 显式（边） | ✓ | $O(|E|)$ |

### 扩展：

- **自注意力**：具有可学习聚合的 RN 特例
- **Transformer**：注意力可以看作关系推理
- **图神经网络**：图结构上的 RN
- **关系型 LSTM**：RN 与循环结构结合

### 限制：

- $O(n^2)$ 复杂度，对较大的 $n$ 计算成本很高
- 求和聚合可能会丢失信息
- 需要对象提取（对于图像来说并非易事）

### 应用：

- 视觉问答
- 物理预测
- 多智能体系统
- 图推理
- 关系数据库
- 任何涉及结构化对象的任务！